# Phase 0: Research Contract

## 0.1 Prediction Task and Evaluation Metric

### Prediction Target

Use the provided market data to predict `responder_6` using only information available at prediction time.

### Unit of Observation

Each row represents one observation for an anonymized financial instrument at a specific date and time step.

- `date_id`: an anonymized chronological date index
- `time_id`: an ordered time step within a date
- `symbol_id`: an anonymized financial instrument identifier

### Sample Weight

The `weight` column determines how strongly each observation contributes to the evaluation score.

### Evaluation Metric

The competition uses the sample-weighted, zero-mean $R^2$ score:

$$
R^2 = 1 - \frac{\sum_i w_i(y_i-\hat{y}_i)^2}{\sum_i w_i y_i^2}
$$

where $y_i$ is the observed `responder_6`, $\hat{y}_i$ is its prediction, and $w_i$ is the sample weight. Predicting zero for every row produces a score of 0; a poor model can produce a negative score.

## 0.2 Information Availability and Validation Timeline

- **Prediction time:** A prediction must be made before the current `responder_6` is revealed.
- **Available information:** Current row identifiers and features are available. Previous-date responders may be used only after their lagged values are supplied by the evaluation API at the first time step of the next date.
- **Unavailable information:** Current or future responders, future features, and preprocessing fitted on future data are unavailable.
- **Validation timeline:** Train on earlier dates, validate on later dates, and reserve the most recent period as a sealed final holdout.

# Phase 1: Data Inventory and Structural Validation

The goal of this phase is to verify the physical layout, time coverage, schema consistency, key uniqueness, and row ordering before any exploratory analysis or modeling.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

project_root = Path.cwd()
if not (project_root / "train.parquet").exists():
    project_root = project_root.parent

train_dir = project_root / "train.parquet"

print("Project root:", project_root.name)
print("Training directory:", train_dir.relative_to(project_root))
print("Exists:", train_dir.exists())
print("Is directory:", train_dir.is_dir())

In [ ]:
partition_files = sorted(
    train_dir.glob("partition_id=*/part-*.parquet")
)

print("Partition count:", len(partition_files))
for file_path in partition_files:
    print(file_path.relative_to(project_root))

In [ ]:
total_size_gib = 0

for file_path in partition_files:
    file_size_gib = file_path.stat().st_size / (1024**3)
    print(f"{file_path.name}: {file_size_gib:.2f} GiB")
    total_size_gib += file_size_gib

print(f"Total size: {total_size_gib:.2f} GiB")

In [ ]:
first_partition = partition_files[0]
parquet_file = pq.ParquetFile(first_partition)
metadata = parquet_file.metadata

print("File:", first_partition.name)
print("Rows:", metadata.num_rows)
print("Row groups:", metadata.num_row_groups)
print("Columns:", metadata.num_columns)
print("First 10 column names:", parquet_file.schema.names[:10])

In [ ]:
inventory = []

for file_path in partition_files:
    parquet_file = pq.ParquetFile(file_path)
    metadata = parquet_file.metadata

    inventory.append(
        {
            "file": file_path.name,
            "rows": metadata.num_rows,
            "row_groups": metadata.num_row_groups,
            "columns": metadata.num_columns,
            "size_gib": round(file_path.stat().st_size / (1024**3), 2),
        }
    )

inventory_df = pd.DataFrame(inventory)
inventory_df

In [ ]:
print(f"Total rows: {inventory_df['rows'].sum():,}")
print(f"Total size: {inventory_df['size_gib'].sum():.2f} GiB")

In [ ]:
date_ranges = []

for file_path in partition_files:
    date_data = pd.read_parquet(file_path, columns=["date_id"])

    date_ranges.append(
        {
            "file": file_path.name,
            "min_date": date_data["date_id"].min(),
            "max_date": date_data["date_id"].max(),
            "unique_dates": date_data["date_id"].nunique(),
        }
    )

date_ranges_df = pd.DataFrame(date_ranges)
date_ranges_df

In [ ]:
reference_schema = pq.ParquetFile(partition_files[0]).schema_arrow

schema_checks = []
for file_path in partition_files:
    current_schema = pq.ParquetFile(file_path).schema_arrow
    schema_checks.append(
        {
            "file": file_path.name,
            "schema_matches": current_schema.equals(reference_schema),
        }
    )

schema_checks_df = pd.DataFrame(schema_checks)
schema_checks_df

In [ ]:
column_types_df = pd.DataFrame(
    {
        "variable": [field.name for field in reference_schema],
        "type": [str(field.type) for field in reference_schema],
    }
)

type_summary_df = (
    column_types_df.groupby("type", as_index=False)["variable"]
    .count()
    .rename(columns={"variable": "count"})
)
type_summary_df["percentage"] = (
    type_summary_df["count"] / len(reference_schema) * 100
).round(2)

type_summary_df

In [ ]:
duplicate_checks = []
key_columns = ["date_id", "time_id", "symbol_id"]

for file_path in partition_files:
    primary_key_data = pd.read_parquet(file_path, columns=key_columns)
    duplicate_rows = primary_key_data.duplicated(subset=key_columns).sum()

    duplicate_checks.append(
        {
            "file": file_path.name,
            "rows": len(primary_key_data),
            "duplicate_rows": duplicate_rows,
        }
    )

duplicate_checks_df = pd.DataFrame(duplicate_checks)
duplicate_checks_df

In [ ]:
time_checks = []

for file_path in partition_files:
    date_time_data = pd.read_parquet(
        file_path,
        columns=["date_id", "time_id"],
    )
    date_time_index = pd.MultiIndex.from_frame(date_time_data)

    time_checks.append(
        {
            "file": file_path.name,
            "sorted_by_date_and_time": date_time_index.is_monotonic_increasing,
        }
    )

time_checks_df = pd.DataFrame(time_checks)
time_checks_df

## 1.1 Verified Findings

- The training data contains 10 Parquet partitions, 47,127,338 rows, and approximately 11.45 GiB.
- Each partition has 92 columns: 4 identifier/weight columns, 79 features, and 9 responders.
- All 10 partitions have identical Arrow schemas.
- `date_id` covers the continuous range 0–1698, with no gaps or overlaps between partitions.
- Within every partition, `(date_id, time_id, symbol_id)` uniquely identifies a row.
- Every partition is monotonically ordered by `date_id` and then `time_id`.
- Later phases should read one partition and only the required columns at a time instead of loading the full dataset into memory.